# 🧬 Borzoi Tools Example

This notebook demonstrates how to run **Borzoi** single-replicate and ensemble prediction tools.

## 📖 What is Borzoi?

[Borzoi](https://www.nature.com/articles/s41588-024-02053-6) is a long-context regulatory sequence model with 524,288 bp input windows and replicate-specific checkpoints.

### Key Features:
- 🧪 **Single-replicate prediction** - Fast run with one selected checkpoint replicate
- 📚 **Ensemble prediction** - Runs all 4 replicates for robust consensus analysis
- 🐭 **Species support** - Human and mouse model heads
- ⚡ **GPU-ready** - Local standalone venv execution and the cloud runtime support

## 📦 Imports


## Input/Output Schema

### `BorzoiInput`
| Field | Type | Description |
|-------|------|-------------|
| sequence | str | DNA sequence to score. Must be exactly 524,288 bp and contain only valid nucleotide characters (A/C/G/T/N). |

### `BorzoiConfig` (single-replicate)
| Field | Type | Default | Description |
|-------|------|---------|-------------|
| output_tracks | List[int] | *(required)* | Track indices to extract from model output |
| species | Literal["human", "mouse"] | "human" | Species model to use |
| replicate | Literal["0", "1", "2", "3"] | "0" | Replicate ID to run |
| avg_output_tracks | bool | True | Whether to average selected tracks into one output |
| use_flash_attn | bool | True | Whether to use FlashAttention models |
| keep_on_gpu | bool | False | Whether to keep model loaded on device after prediction |
| device | str | "cuda" | Device to run the model on |
| verbose | bool | False | Whether to print status messages during execution |

### `BorzoiOutput` (single-replicate)
| Field | Type | Description |
|-------|------|-------------|
| sequence | str | Input DNA sequence that was scored |
| sequence_length | int | Length of the input sequence (always 524,288) |
| prediction | List[List[float]] | Prediction matrix with shape [num_tracks, 6144] |
| output_tracks | List[int] | Track indices used for prediction |
| species | str | Species used for prediction ('human' or 'mouse') |
| replicate | str | Replicate used for prediction ('0' to '3') |
| avg_output_tracks | bool | Whether track outputs were averaged |

### `BorzoiEnsembleConfig`
| Field | Type | Default | Description |
|-------|------|---------|-------------|
| output_tracks | List[int] | *(required)* | Track indices to extract from model output |
| species | Literal["human", "mouse"] | "human" | Species model to use |
| avg_output_tracks | bool | True | Whether to average selected tracks into one output |
| use_flash_attn | bool | True | Whether to use FlashAttention models |
| keep_on_gpu | bool | False | Whether to keep model loaded on device after prediction |
| device | str | "cuda" | Device to run the model on |
| verbose | bool | False | Whether to print status messages during execution |

### `BorzoiEnsembleOutput`
| Field | Type | Description |
|-------|------|-------------|
| sequence | str | Input DNA sequence that was scored |
| sequence_length | int | Length of the input sequence (always 524,288) |
| predictions | List[List[List[float]]] | Stacked predictions with shape [4, num_tracks, 6144] for replicates 0-3 |
| output_tracks | List[int] | Track indices used for prediction |
| species | str | Species used for prediction ('human' or 'mouse') |
| avg_output_tracks | bool | Whether track outputs were averaged |
| num_replicates | int | Number of Borzoi replicates returned (always 4) |

In [1]:
import numpy as np
from bio_programming.bio_tools.tools.sequence_scoring.borzoi import (
    BORZOI_CONTEXT,
    BorzoiInput,
    BorzoiConfig,
    BorzoiEnsembleConfig,
    run_borzoi,
    run_borzoi_ensemble,
)

## 🧪 1. Single-Replicate Prediction (`run_borzoi`)

Run one Borzoi replicate and return a `[num_tracks, 6144]` prediction matrix.

### Configuration options:
- 🧠 **`output_tracks`** - Track indices to extract
- 🧬 **`replicate`** - Replicate ID (`"0"` to `"3"`)
- 🐭 **`species`** - `human` or `mouse`
- 📉 **`avg_output_tracks`** - Average selected tracks into one signal


In [2]:
sequence = "ATCG" * (BORZOI_CONTEXT // 4)

inputs = BorzoiInput(sequence=sequence)
single_config = BorzoiConfig(
    output_tracks=[0, 1, 2],
    species="human",
    replicate="0",
    avg_output_tracks=False,
    verbose=False,
)

single_result = run_borzoi(inputs, single_config)
print(f"Tracks: {len(single_result.prediction)}")
print(f"Bins per track: {len(single_result.prediction[0])}")


Tracks: 3
Bins per track: 6144


## 📚 2. Ensemble Prediction (`run_borzoi_ensemble`)

Run all 4 Borzoi replicates and stack outputs for consensus statistics.


In [3]:
ensemble_config = BorzoiEnsembleConfig(
    output_tracks=[0, 1, 2],
    species="human",
    avg_output_tracks=True,
    verbose=False,
)

ensemble_result = run_borzoi_ensemble(inputs, ensemble_config)
ensemble_array = np.array(ensemble_result.predictions)
mean_pred = ensemble_array.mean(axis=0)
std_pred = ensemble_array.std(axis=0)
print(f"Replicates: {ensemble_array.shape[0]}")
print(f"Mean shape: {mean_pred.shape}")
print(f"Std max: {std_pred.max():.4f}")


Replicates: 4
Mean shape: (1, 6144)
Std max: 0.0051


## 💾 3. Export Results

Save single-replicate and ensemble outputs to the example output directory.

### Supported formats:
- 📄 **JSON**
- 📊 **CSV**


In [4]:
single_result.export("example_output/borzoi_single", file_format="json")
print("Single-replicate result exported to example_output/borzoi_single.json")

ensemble_result.export("example_output/borzoi_ensemble", file_format="csv")
print("Ensemble summary exported to example_output/borzoi_ensemble.csv")


Single-replicate result exported to example_output/borzoi_single.json
Ensemble summary exported to example_output/borzoi_ensemble.csv
